# 04. Broadcasting Rules & Matrix Centering: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **04. Broadcasting Rules & Matrix Centering**. Broadcasting is NumPy's powerful mechanism for executing arithmetic operations between arrays of different shapes without copying data in memory. This notebook covers the formal trailing dimension compatibility rules, explicit dimension expansion with `np.newaxis` and `None`, explicit broadcasting utilities (`np.broadcast_to`, `np.broadcast_arrays`), and 2D/3D coordinate grid generation (`np.meshgrid`, `np.mgrid`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Trailing Dimension Compatibility Rules
- [x] 🔹 Feature Normalization (Z-Score) via Broadcasting
- [x] 🔹 Dimension Expansion with `np.newaxis`
- [x] 🔹 Dimension Expansion with `None`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 Trailing Dimension Compatibility Rules
- **What it does:** Broadcasting 1D feature mean vector across 2D elements matrix.
- **Syntax:** `function(*args, **kwargs)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Trailing Dimension Compatibility Rules across the extracted numeric transaction `amounts` array to compute performance metrics.


In [2]:
tx_mat = np.column_stack([amounts[:1000], account_ages[:1000]])
col_means = tx_mat.mean(axis=0)
centered_mat = tx_mat - col_means  # (1000, 2) - (2,) -> Broadcasts across rows
print('Matrix Shape:', tx_mat.shape, 'Mean Vector Shape:', col_means.shape)
print('Centered Matrix Head:\n', centered_mat[:3].round(2))

Matrix Shape: (1000, 2) Mean Vector Shape: (2,)
Centered Matrix Head:
 [[-409.58  -52.75]
 [ 801.75  -32.75]
 [-953.28   30.25]]


### 🔹 Feature Normalization (Z-Score) via Broadcasting
- **What it does:** Normalizes multi-feature elements matrix to mean 0 and std 1.
- **Syntax:** `function(*args, **kwargs)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Two dimensions are compatible for broadcasting if they are equal, or if one of them is 1. Dimensions are matched from right to left.
- **Dataset Application & Code Demonstration:** Demonstrates Feature Normalization (Z-Score) via Broadcasting with practical fintech data structures and variables in the following code block.


In [3]:
col_stds = tx_mat.std(axis=0)
z_scores = (tx_mat - col_means) / col_stds
print('Z-Scored Feature Columns (Head):\n', z_scores[:3].round(2))

Z-Scored Feature Columns (Head):
 [[-0.71 -1.5 ]
 [ 1.39 -0.93]
 [-1.65  0.86]]


### 🔹 Dimension Expansion with `np.newaxis`
- **What it does:** Increases the dimension of the existing array by one unit dimension (1D vector of length N becomes 2D matrix of shape Nx1 or 1xN).
- **Syntax:** `arr[:, np.newaxis] / arr[np.newaxis, :]`
- **Key Note:** Enables broadcasting between arrays of incompatible dimensionalities.
- **Dataset Application & Code Demonstration:** Applies Dimension Expansion with `np.newaxis` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [4]:
col_vector = amounts[:5, np.newaxis]
print('1D Amounts converted to Column Vector Shape:', col_vector.shape)

1D Amounts converted to Column Vector Shape: (5, 1)


### 🔹 Dimension Expansion with `None`
- **What it does:** Aliases `None` to add batch dimension `(1, N)`.
- **Syntax:** `None`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Dimension Expansion with `None` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [5]:
row_vector = amounts[:5][None, :]
print('1D Amounts converted to Row Vector Shape:', row_vector.shape)

1D Amounts converted to Row Vector Shape: (1, 5)


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Softmax Implementation on Risk Logits
- **Objective:** Q1: Softmax Implementation on Risk Logits
- **Approach:** Vectorize softmax probability estimation for multi-class transaction classification.
- **Syntax:** `np.exp(logits - np.max(logits, axis=1, keepdims=True))`

In [6]:
mock_logits = np.column_stack([amounts[:5]/100, account_ages[:5]/10])
shifted = mock_logits - np.max(mock_logits, axis=1, keepdims=True)
probs = np.exp(shifted) / np.sum(np.exp(shifted), axis=1, keepdims=True)
print('Softmax Output Probabilities:\n', probs.round(3))

Softmax Output Probabilities:
 [[0.995 0.005]
 [1.    0.   ]
 [0.    1.   ]
 [0.995 0.005]
 [0.999 0.001]]
